# Week 5: SQL practice

Remember to save a copy with `_mine` in the filename.

## Setup

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd

# The course data folder is two levels up from this notebook.
DATA = Path("../../data")
if not DATA.exists():  # Colab: the course folder sits in /content
    DATA = Path("/content/compss-211a/data")

In [ ]:
# One row per San Francisco 311 request, August 3-9, 2026.
# IDs are read as text. district is a whole number that can be missing.
raw = pd.read_csv(DATA / "sf311_requests.csv", dtype={"request_id": "string", "district": "Int64"})

# Leave out the rows whose channel is Test.
requests = raw[raw["channel"] != "Test"]

print(len(raw), "rows in the file,", len(requests), "kept")
print("Every request_id appears once:", requests["request_id"].is_unique)
requests.head(3)

In [ ]:
# One row per supervisor district. Keep two columns and give them shorter names.
districts = pd.read_csv(DATA / "sf_supervisor_districts.csv")
districts = districts.rename(columns={"sup_dist": "district", "sup_dist_name": "district_name"})
districts = districts[["district", "district_name"]]
districts

In [ ]:
# New this week: copy both tables into a database.
# This one lives in memory and disappears when the notebook closes.
database = sqlite3.connect(":memory:")
requests.to_sql("requests", database, index=False)
districts.to_sql("districts", database, index=False)

# sql("...") sends a query to the database and returns the answer as a pandas table.
def sql(query):
    return pd.read_sql_query(query, database)

## 1. Select and filter

The first cell asks one question twice, in pandas and in SQL. Then fill the blank to show five Phone requests.

In [ ]:
# pandas, which you know: three columns, sorted by ID, first five rows.
in_pandas = requests[["request_id", "category", "district"]].sort_values("request_id").head(5)

# SQL: the same request.
in_sql = sql("""
    SELECT request_id, category, district
    FROM requests
    ORDER BY request_id
    LIMIT 5
""")

display(in_pandas, in_sql)

In [ ]:
# In pandas you would write requests[requests["channel"] == "Phone"]. In SQL that is WHERE.
sql("""
    SELECT request_id, category, district
    FROM requests
    WHERE channel = '____'
    ORDER BY request_id
    LIMIT 5
""")

## 2. Count groups

Run the channel example. Then complete the query: how many Phone requests came from each district? Show the largest count first.

In [ ]:
# Worked example: one row per reporting channel.
sql("""
    SELECT channel, COUNT(*) AS requests
    FROM requests
    GROUP BY channel
    ORDER BY requests DESC, channel ASC
""")

In [ ]:
sql("""
    SELECT ____, COUNT(*) AS requests
    FROM requests
    WHERE channel = 'Phone'
    GROUP BY ____
    ORDER BY requests ____, district ASC
""")

## 3. Find missing values

Run the counts example. Then complete the query to count requests with an unknown district. Hint: `IS NULL`.

In [ ]:
# Worked example: compare all rows with observed durations.
sql("""
    SELECT COUNT(*) AS requests,
           COUNT(hours_to_close) AS with_time,
           AVG(hours_to_close) AS mean_hours
    FROM requests
""")

In [ ]:
sql("""
    SELECT COUNT(*) AS requests
    FROM requests
    WHERE district ____
""")

## 4. Join two tables

Match requests to district names using `district`. Keep every request. Fill both blanks, then run the check.

In [ ]:
joined = sql("""
    SELECT r.request_id, r.category, r.district, d.district_name
    FROM requests AS r
    LEFT JOIN districts AS d
        ON r.____ = d.____
    ORDER BY r.request_id
""")
joined.head()

In [ ]:
print("Before:", len(requests), "After:", len(joined))
print("Unique request IDs:", joined["request_id"].nunique())
print("Unmatched requests:", joined["district_name"].isna().sum())
assert len(joined) == len(requests)
assert joined["request_id"].notna().all() and joined["request_id"].is_unique
assert set(joined["request_id"]) == set(requests["request_id"])

## 5. Write your own SQL

Start from an empty cell; earlier examples are available for reference.

1. Show the five categories with the most **Open** requests, with their counts.
2. Choose a channel. Count its requests by **district name**, keeping requests without a match. This needs both tables.

In [ ]:
# Write query 1 here.

In [ ]:
# Write query 2 here.

## 6. Food access by county

20 minutes, on your own. A food-access nonprofit wants to know which California counties to look at first.

`tracts` has one row per census tract, a small area with a few thousand residents. `counties` has one row per county. `low_income_low_access` is 1 when USDA flags a tract as low-income and far from a store that accepts SNAP benefits. Otherwise it is 0. [USDA definitions](https://www.ers.usda.gov/data-products/food-access-research-atlas/documentation/snap-authorized-retailer-access-map-reference-guide).

In [ ]:
# Load the two new tables and add them to the database. County codes stay text.
tracts = pd.read_csv(DATA / "week05_food_access_tracts.csv", dtype={"tract_id": "string", "county_fips": "string"})
counties = pd.read_csv(DATA / "week05_ca_counties.csv", dtype={"county_fips": "string"})
tracts.to_sql("tracts", database, index=False, if_exists="replace")
counties.to_sql("counties", database, index=False, if_exists="replace")

display(tracts.head(3), counties.head(3))

In [ ]:
# 1. Count the flagged tracts for each county name. Largest count first.
#    You need both tables. Name the columns county_name and flagged.
by_count = ...

In [ ]:
# Check: the counts add up to every flagged tract in the state.
all_flagged = sql("SELECT COUNT(*) AS tracts FROM tracts WHERE low_income_low_access = 1")
assert by_count["flagged"].sum() == all_flagged.loc[0, "tracts"]
by_count.head(5)

In [ ]:
# The flag is 1 or 0, so its sum is the number of flagged tracts
# and its average is the share of tracts that are flagged.
# Worked example: the whole state.
sql("""
    SELECT COUNT(*) AS tracts,
           SUM(low_income_low_access) AS flagged,
           AVG(low_income_low_access) AS share
    FROM tracts
""")

In [ ]:
# 2. Will the county with the most flagged tracts also have the largest share?
#    My guess:
#
#    Now make the same three numbers for each county name. Largest share first.
#    Name the columns county_name, tracts, flagged, and share.
by_share = ...

In [ ]:
# Check: one row per county, and the same flagged tracts as before.
assert len(by_share) == len(counties)
assert by_share["flagged"].sum() == by_count["flagged"].sum()

# The two top fives. Which list should the nonprofit see? We will discuss.
display(by_count.head(5), by_share.head(5))